# 🎵 Spectral Struct — End-to-End Feature Extraction & XGBoost Training
This notebook has been completely overhauled to ensure maximum stability and strict compliance with the plan:
1. **Download dataset via curl** exactly as requested.
2. **Save tabular columns** to Google Drive safely.
3. **Train XGBoost model** strictly on `train` and evaluate on `test`.
4. **Max CPU computation power** (`n_jobs=-1`).
5. **Multithreading** for rapid feature extraction.

*Note: We only use the `train` and `test` splits (val is discarded), and we sample 40% of the files. Robust error handling is added to prevent crashes.*

## 1. Environment Setup & Mount Drive

In [ ]:
!pip install -q librosa soundfile pandas tqdm xgboost scikit-learn

import os
import sys
import subprocess
from pathlib import Path
import warnings
import random
import time
import json
import concurrent.futures

import numpy as np
import pandas as pd
import librosa
import xgboost as xgb
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score, classification_report
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OUT_DIR = Path('/content/drive/MyDrive/MAD_Project/spectral_struct')
except ImportError:
    print("Not running in Colab. Using local directory.")
    DRIVE_OUT_DIR = Path('./MAD_Project/spectral_struct')

DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[✔] Output directory set to: {DRIVE_OUT_DIR}")


## 2. Dataset Download using `curl`
Using the exact curl command requested to download the dataset, followed by unzipping.

In [ ]:
DATA_DIR = Path('/content/data/asvspoof5')
ZIP_PATH = Path('/content/asvspoof5-flac.zip')

if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    print(f'[✔] Dataset already exists at {DATA_DIR}')
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'[↓] Downloading dataset using curl...')
    
    # Note: Kaggle API requires authentication. If this curl command downloads a small HTML file instead of the zip,
    # it is because Kaggle requires authentication cookies or tokens. 
    # If that happens, please use the kaggle CLI: !kaggle datasets download -d aniket202411001/asvspoof5-flac
    !curl -L -o {ZIP_PATH} https://www.kaggle.com/api/v1/datasets/download/aniket202411001/asvspoof5-flac
    
    if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 1000000: # Ensure it's not a tiny HTML login page
        print(f'[↓] Unzipping dataset...')
        !unzip -q {ZIP_PATH} -d {DATA_DIR}
        ZIP_PATH.unlink()  # Remove zip file to save space
        print('[✔] Download and extraction complete.')
    else:
        print("[!] Downloaded file is too small or missing. This usually means Kaggle rejected the unauthenticated curl request.")
        print("    Please authenticate or use the Kaggle API to download the dataset.")


## 3. Feature Extraction Definitions
Extracting ~110 spectral features.

In [ ]:
SR = 16000

def extract_spectral_row(path, sr=SR):
    y, _ = librosa.load(str(path), sr=sr, mono=True)
    peak = np.abs(y).max()
    if peak > 0: y = y / peak
    
    row = {}
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=512, hop_length=160)
    for i in range(20):
        row[f'mfcc{i}_mean'] = float(mfcc[i].mean())
        row[f'mfcc{i}_std']  = float(mfcc[i].std())
        
    dmfcc = librosa.feature.delta(mfcc)
    for i in range(20):
        row[f'dmfcc{i}_mean'] = float(dmfcc[i].mean())
        row[f'dmfcc{i}_std']  = float(dmfcc[i].std())
    
    for name, fn in [('centroid', librosa.feature.spectral_centroid),
                     ('bandwidth', librosa.feature.spectral_bandwidth),
                     ('rolloff', librosa.feature.spectral_rolloff),
                     ('flatness', librosa.feature.spectral_flatness)]:
        feat = fn(y=y, sr=sr)[0]
        row[f'{name}_mean'] = float(feat.mean())
        row[f'{name}_std']  = float(feat.std())
        row[f'{name}_max']  = float(feat.max())
    
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        row[f'contrast{i}_mean'] = float(contrast[i].mean())
        row[f'contrast{i}_std']  = float(contrast[i].std())
    
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    row['zcr_mean'] = float(zcr.mean())
    row['zcr_std']  = float(zcr.std())
    
    rms = librosa.feature.rms(y=y)[0]
    row['rms_mean'] = float(rms.mean())
    row['rms_std']  = float(rms.std())
    row['rms_max']  = float(rms.max())
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12):
        row[f'chroma{i}_mean'] = float(chroma[i].mean())
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    row['mel_mean']  = float(mel_db.mean())
    row['mel_std']   = float(mel_db.std())
    row['mel_max']   = float(mel_db.max())
    row['mel_min']   = float(mel_db.min())
    
    return row


## 4. Multithreaded Dataset Construction (Train & Test, 40%)
Extracts exactly 40% of the train and test splits, completely discarding the validation set.

In [ ]:
DATA_FACTOR = 0.4  # Using exactly 40% of the dataset
TARGET_SPLITS = ['train', 'test']  # Ditching validation set

def resolve_split(root, split):
    d = root / split
    if d.exists(): return d
    mapping = {'train': 'flac_T', 'val': 'flac_D', 'test': 'flac_E_eval'}
    c = root / mapping.get(split, '')
    if c.exists(): return c
    for x in root.iterdir():
        if x.is_dir() and split in x.name.lower(): return x
    return d

def collect_files(root, split, factor):
    sd = resolve_split(root, split)
    bona_dir = sd / 'bonafide'
    spoof_dir = sd / 'spoof'
    
    bona = list(bona_dir.glob('**/*.flac')) + list(bona_dir.glob('**/*.wav')) if bona_dir.exists() else []
    spoof = list(spoof_dir.glob('**/*.flac')) + list(spoof_dir.glob('**/*.wav')) if spoof_dir.exists() else []
    
    # Safely sample exactly 40%
    if len(bona) > 0:
        bona = random.sample(bona, min(len(bona), max(1, int(len(bona) * factor))))
    if len(spoof) > 0:
        spoof = random.sample(spoof, min(len(spoof), max(1, int(len(spoof) * factor))))
    
    files = [(f, 0) for f in bona] + [(f, 1) for f in spoof]  
    print(f'  [{split}] bonafide={len(bona)}, spoof={len(spoof)}, total={len(files)} (sampled 40%)')
    return files

def process_file(file_info):
    path, label = file_info
    try:
        row = extract_spectral_row(path)
        row['label']    = label
        row['filename'] = path.name
        return row
    except Exception as e:
        return None

t0 = time.time()
for split in TARGET_SPLITS:
    out_csv = DRIVE_OUT_DIR / f'{split}.csv'
    
    if out_csv.exists():
        print(f'[{split}] Already exists at {out_csv} — skipping extraction.')
        continue
        
    print(f'\n[{split}] Collecting files...')
    files = collect_files(DATA_DIR, split, DATA_FACTOR)
    
    if not files:
        print(f'  [{split}] No files found. Ensure the dataset is extracted correctly.')
        continue

    max_workers = os.cpu_count() or 4
    print(f'  [{split}] Extracting features using {max_workers} threads (Multithreading)...')
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(process_file, files), total=len(files), desc=f'Extracting {split}'))
        
    rows = [r for r in results if r is not None]
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f'  [{split}] Saved {len(df)} rows × {len(df.columns)} cols → {out_csv}')

print(f'\n[✔] Feature extraction completed in {(time.time()-t0)/60:.1f} min')


## 5. Optimized XGBoost Training
Trains on `train.csv` and evaluates on `test.csv` using maximum CPU compute.

In [ ]:
def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    try:
        return float(brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1))
    except:
        return float(np.mean(np.abs(fnr - fpr)))

print("[⚙] Loading datasets...")
try:
    train_df = pd.read_csv(DRIVE_OUT_DIR / 'train.csv')
    test_df  = pd.read_csv(DRIVE_OUT_DIR / 'test.csv')
except FileNotFoundError as e:
    print(f"[!] Error loading CSV: {e}. Please ensure extraction completed successfully.")
    raise

feat_cols = [c for c in train_df.columns if c not in ('label', 'filename')]

X_train = train_df[feat_cols].values
y_train = train_df['label'].values
X_test = test_df[feat_cols].values
y_test = test_df['label'].values

np.nan_to_num(X_train, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
np.nan_to_num(X_test, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")

print("\n[⚙] Initializing XGBoost Classifier (Max Compute)...")
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1,  # Uses all CPU cores available
    random_state=42
)

print("[⚙] Training...")
model.fit(X_train, y_train, verbose=50)

print("\n[⚙] Evaluating on Test Set...")
proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

eer = compute_eer(y_test, proba)
fpr, tpr, _ = roc_curve(y_test, proba, pos_label=1)
auc_val = auc(fpr, tpr)
acc = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)

print(f"{'='*40}")
print(f"Test EER:      {eer*100:.2f}%")
print(f"Test AUC:      {auc_val:.4f}")
print(f"Test Accuracy: {acc*100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")
print(f"{'='*40}")
print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=["bonafide", "spoof"]))

model_path = DRIVE_OUT_DIR / "best_xgb.json"
model.save_model(str(model_path))
print(f"\n[✔] Model saved to {model_path}")

with open(DRIVE_OUT_DIR / "feature_cols.json", "w") as f:
    json.dump(feat_cols, f)

with open(DRIVE_OUT_DIR / "results.txt", "w") as f:
    f.write(f"EER: {eer*100:.4f}%\nAUC: {auc_val:.4f}\nAccuracy: {acc*100:.4f}%\nF1: {f1:.4f}\n")

with open(DRIVE_OUT_DIR / "results.json", "w") as f:
    json.dump({"eer": eer, "auc": auc_val, "accuracy": acc, "f1": f1}, f, indent=4)

print("[✔] Results saved successfully.")
